In [295]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split

from imblearn.under_sampling import RandomUnderSampler

from collections import Counter

from sklearn.metrics import f1_score, average_precision_score, roc_auc_score


In [296]:
link = '/Users/maxkucher/preprocessing/mlops/customer_churn/churn_data.csv'
data = pd.read_csv(link)
data

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [297]:
def clean_data(data):
    data = data.copy()

    data['MultipleLines'] = data['MultipleLines'].replace({'No phone service': 'No'})

    columns = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies']
    
    for col in columns:
        data[col] = data[col].replace({'No internet service': 'No'})
    
    return data

cleaner = FunctionTransformer(clean_data)

In [298]:
data['TotalCharges'] = data['TotalCharges'].replace({' ': np.nan})
data = data.dropna(subset=['TotalCharges'])

In [299]:
num_columns = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [300]:
numeric_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', MinMaxScaler())
])

In [301]:
binary_columns = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies']

In [302]:
binary_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Yes', 'No']] * len(binary_columns)))
])

In [303]:
cat_columns = ['InternetService', 'Contract', 'PaymentMethod']

In [304]:
categorical_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [305]:
gender_columns = ['gender']

In [306]:
gender_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Female', 'Male']]))
])

In [307]:
transformer = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_columns),
    ('bin', binary_pipeline, binary_columns),
    ('gen', gender_pipeline, gender_columns),
    ('cat', categorical_pipeline, cat_columns)
])

In [308]:
model_pipeline = Pipeline(steps=[
    ('cleaner', cleaner),
    ('transformer', transformer),
    ('model', LogisticRegression(max_iter=100, penalty='l2', C=0.1))
])

In [309]:
model_pipeline

Pipeline(steps=[('cleaner',
                 FunctionTransformer(func=<function clean_data at 0x28d593ce0>)),
                ('transformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   MinMaxScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('bin',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('e...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['Female',
                                                                                               'Male']]))]),
                                                  ['gender']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['InternetService',
                                                   'Contract',
                                                   'PaymentMethod'])])),
                ('model', LogisticRegression(C=0.1))])

In [310]:
data = data.drop('customerID', axis='columns')

In [311]:
x = data.drop('Churn', axis='columns')
y = data['Churn']

In [312]:
rus = RandomUnderSampler()
x, y = rus.fit_resample(x, y)
y = y.map({'No': 0, 'Yes': 1})

In [313]:
Counter(y)

Counter({0: 1869, 1: 1869})

In [314]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [315]:
model_pipeline.fit(x_train, y_train)

Pipeline(steps=[('cleaner',
                 FunctionTransformer(func=<function clean_data at 0x28d593ce0>)),
                ('transformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   MinMaxScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('bin',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('e...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['Female',
                                                                                               'Male']]))]),
                                                  ['gender']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['InternetService',
                                                   'Contract',
                                                   'PaymentMethod'])])),
                ('model', LogisticRegression(C=0.1))])

In [319]:
test_pred = model_pipeline.predict(x_test)
test_prob = model_pipeline.predict_proba(x_test)[:, 1]

train_pred = model_pipeline.predict(x_train)
train_prob = model_pipeline.predict_proba(x_train)[:, 1]

In [320]:
test_f1 = f1_score(test_pred, y_test)
test_pr_auc = average_precision_score(y_test, test_prob)
test_roc_auc = roc_auc_score(y_test, test_prob)

train_f1 = f1_score(train_pred, y_train)
train_pr_auc = average_precision_score(y_train, train_prob)
train_roc_auc = roc_auc_score(y_train, train_prob)

print(f"[TRAIN] F1-score: {train_f1} | PR-AUC: {train_pr_auc} | ROC-AUC: {train_roc_auc}")
print(f"[TEST] F1-score: {test_f1} | PR-AUC: {test_pr_auc} | ROC-AUC: {test_roc_auc}")



[TRAIN] F1-score: 0.7646109137875363 | PR-AUC: 0.8081213711845848 | ROC-AUC: 0.8365171389912051
[TEST] F1-score: 0.8035264483627205 | PR-AUC: 0.8515487565900614 | ROC-AUC: 0.8573932954098243
